In [1]:
# Install required packages first
!pip install -q transformers accelerate>=1.0.0 datasets>=3.0.0
!pip install -q av librosa soundfile resampy
!pip install -q scikit-learn matplotlib seaborn
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pytorchvideo
!pip install -q evaluate
!pip install -q datasets

Found existing installation: torch 2.7.1+cu118
Uninstalling torch-2.7.1+cu118:
  Successfully uninstalled torch-2.7.1+cu118
Found existing installation: torchvision 0.22.1+cu118
Uninstalling torchvision-0.22.1+cu118:
  Successfully uninstalled torchvision-0.22.1+cu118
Found existing installation: torchaudio 2.7.1+cu118
Uninstalling torchaudio-2.7.1+cu118:
  Successfully uninstalled torchaudio-2.7.1+cu118


In [2]:
import sys
import os

dummy_path = "/kaggle/working/notebook.py"

# tạo file giả nếu chưa có
if not os.path.exists(dummy_path):
    with open(dummy_path, "w") as f:
        f.write("# dummy file")

# gán _file_ cho notebook
sys.modules["__main__"].__file__ = dummy_path

In [3]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import os
import gc
import cv2
import warnings
from typing import Dict, List, Optional, Tuple, Union
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score, precision_score
from dataclasses import dataclass
import time
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr
import io
# Transformers components
from transformers import (
    AutoTokenizer, AutoModel, AutoImageProcessor,
    TimesformerForVideoClassification, Trainer, TrainingArguments, 
    EvalPrediction, PreTrainedModel, PretrainedConfig
)
from torch.utils.data import Dataset, DataLoader
import evaluate
from sentence_transformers import SentenceTransformer

In [4]:
# =============================================================================
# COMPLETE ERROR SUPPRESSION AND ENVIRONMENT SETUP
# =============================================================================

def setup_ultimate_silent_environment():
    """Setup the most comprehensive silent environment"""
    
    # Environment variables for complete silence
    silent_vars = {
        'OPENCV_LOG_LEVEL': 'SILENT',
        'OPENCV_VIDEOIO_DEBUG': '0',
        'OPENCV_FFMPEG_DEBUG': '0',
        'OPENCV_VIDEOWRITER_DEBUG': '0',
        'OPENCV_VIDEOCAPTURE_DEBUG': '0',
        'OPENCV_FFMPEG_LOGLEVEL': '-8',
        'OPENCV_AVFOUNDATION_SKIP_AUTH': '1',
        'FFREPORT': 'file=/dev/null:level=quiet',
        'WANDB_DISABLED': 'true',
        'TOKENIZERS_PARALLELISM': 'false',
        'PYTHONWARNINGS': 'ignore',
        'CUDA_LAUNCH_BLOCKING': '0'
    }
    
    for key, value in silent_vars.items():
        os.environ[key] = value
    
    # Configure OpenCV for complete silence
    try:
        cv2.setLogLevel(0)
        cv2.setUseOptimized(True)
    except:
        pass
    
    # Suppress all warnings
    warnings.filterwarnings('ignore')
    warnings.simplefilter('ignore')

@contextmanager
def ultimate_stderr_suppression():
    """Most advanced stderr suppression for C libraries"""
    old_stderr = None
    try:
        if hasattr(os, 'devnull'):
            old_stderr = sys.stderr
            sys.stderr = open(os.devnull, 'w')
            yield
        else:
            old_stderr = sys.stderr
            sys.stderr = io.StringIO()
            yield
    except:
        yield
    finally:
        if old_stderr is not None:
            try:
                if hasattr(sys.stderr, 'close'):
                    sys.stderr.close()
            except:
                pass
            sys.stderr = old_stderr

# Setup environment immediately
setup_ultimate_silent_environment()
print("=== SMART MULTIMODAL CLASSIFIER - ADAPTIVE VALIDATION (FIXED) ===")

def cleanup_memory():
    """Enhanced memory cleanup"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

cleanup_memory()

# =============================================================================
# 1. SMART VIDEO VALIDATION SYSTEM (SAME AS BEFORE)
# =============================================================================

class SmartVideoValidator:
    """Smart video validator with adaptive fallback strategies"""
    
    def __init__(self, strict_mode=False):
        self.strict_mode = strict_mode
        self.validation_cache = {}
        self.ffprobe_available = None
        self.stats = {
            'total': 0, 'valid': 0, 'invalid': 0,
            'not_found': 0, 'too_small': 0, 
            'ffprobe_failed': 0, 'opencv_failed': 0,
            'opencv_success': 0, 'ffprobe_success': 0
        }
        
        # Check FFprobe availability
        self._check_ffprobe_availability()
        
        print(f"📋 Validator initialized:")
        print(f"  FFprobe available: {self.ffprobe_available}")
        print(f"  Strict mode: {self.strict_mode}")
    
    def _check_ffprobe_availability(self):
        """Check if FFprobe is available on the system"""
        try:
            with ultimate_stderr_suppression():
                result = subprocess.run(['ffprobe', '-version'], 
                                      capture_output=True, 
                                      timeout=5)
            self.ffprobe_available = (result.returncode == 0)
        except:
            self.ffprobe_available = False
        
        if not self.ffprobe_available:
            print("⚠️ FFprobe not available - will use OpenCV-only validation")
    
    def is_video_valid(self, video_path: str) -> bool:
        """Smart video validation with adaptive strategies"""
        
        if video_path in self.validation_cache:
            return self.validation_cache[video_path]
        
        self.stats['total'] += 1
        is_valid = False
        
        try:
            # Step 1: Basic checks
            if not video_path or not os.path.exists(video_path):
                self.stats['not_found'] += 1
                self.validation_cache[video_path] = False
                return False
            
            # Step 2: File size check (more lenient)
            try:
                file_size = os.path.getsize(video_path)
                if file_size < 1000:  # Less than 1KB
                    self.stats['too_small'] += 1
                    self.validation_cache[video_path] = False
                    return False
            except:
                self.stats['too_small'] += 1
                self.validation_cache[video_path] = False
                return False
            
            # Step 3: Try FFprobe if available, otherwise use OpenCV
            if self.ffprobe_available:
                if self._validate_with_ffprobe(video_path):
                    is_valid = True
                    self.stats['ffprobe_success'] += 1
                else:
                    self.stats['ffprobe_failed'] += 1
                    # If FFprobe fails, try OpenCV as fallback
                    if self._validate_with_opencv(video_path):
                        is_valid = True
                        self.stats['opencv_success'] += 1
                    else:
                        self.stats['opencv_failed'] += 1
            else:
                # OpenCV-only validation
                if self._validate_with_opencv(video_path):
                    is_valid = True
                    self.stats['opencv_success'] += 1
                else:
                    self.stats['opencv_failed'] += 1
                
        except:
            self.stats['invalid'] += 1
        
        if not is_valid:
            self.stats['invalid'] += 1
        else:
            self.stats['valid'] += 1
        
        self.validation_cache[video_path] = is_valid
        return is_valid
    
    def _validate_with_ffprobe(self, video_path: str) -> bool:
        """FFprobe validation with lenient criteria"""
        try:
            cmd = [
                'ffprobe', '-v', 'quiet', '-select_streams', 'v:0',
                '-show_entries', 'stream=width,height,nb_frames',
                '-of', 'csv=p=0', video_path
            ]
            
            with ultimate_stderr_suppression():
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            
            if result.returncode != 0:
                return False
            
            # Parse output
            output = result.stdout.strip()
            if not output:
                return False
            
            try:
                parts = output.split(',')
                if len(parts) >= 2:
                    width = int(parts[0]) if parts[0] else 0
                    height = int(parts[1]) if parts[1] else 0
                    
                    # Very lenient criteria
                    if width > 0 and height > 0:
                        return True
            except (ValueError, IndexError):
                pass
            
            return False
            
        except:
            return False
    
    def _validate_with_opencv(self, video_path: str) -> bool:
        """OpenCV validation with lenient criteria"""
        cap = None
        try:
            with ultimate_stderr_suppression():
                # Try different backends
                for backend in [cv2.CAP_FFMPEG, cv2.CAP_ANY]:
                    try:
                        cap = cv2.VideoCapture(video_path, backend)
                        if cap and cap.isOpened():
                            break
                    except:
                        continue
                
                if not cap or not cap.isOpened():
                    return False
                
                # Get basic properties
                frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                
                # Very lenient criteria - just check if we can get basic properties
                if width > 0 and height > 0:
                    # Try to read at least one frame
                    ret, frame = cap.read()
                    if ret and frame is not None and frame.size > 0:
                        return True
                
                return False
                
        except:
            return False
        finally:
            if cap:
                try:
                    cap.release()
                except:
                    pass
    
    def validate_dataset_sample(self, df: pd.DataFrame, sample_size: int = 100) -> pd.DataFrame:
        """Validate a sample first to check validation success rate"""
        print(f"🧪 Testing validation on {sample_size} samples...")
        
        sample_df = df.head(sample_size) if len(df) > sample_size else df
        valid_count = 0
        
        for idx, row in sample_df.iterrows():
            video_path = row['video_path']
            if self.is_video_valid(video_path):
                valid_count += 1
        
        success_rate = (valid_count / len(sample_df)) * 100
        print(f"📊 Sample validation results:")
        print(f"  Tested: {len(sample_df)} files")
        print(f"  Valid: {valid_count}")
        print(f"  Success rate: {success_rate:.1f}%")
        
        if success_rate < 10:
            print("⚠️ Very low validation success rate!")
            print("🔄 Switching to lenient OpenCV-only mode...")
            return self._validate_dataset_lenient(df)
        else:
            return self._validate_dataset_normal(df)
    
    def _validate_dataset_normal(self, df: pd.DataFrame) -> pd.DataFrame:
        """Normal validation process"""
        print("🔍 Performing normal video validation...")
        
        valid_indices = []
        total = len(df)
        
        for idx, row in df.iterrows():
            video_path = row['video_path']
            if self.is_video_valid(video_path):
                valid_indices.append(idx)
            
            if (idx + 1) % 500 == 0:
                print(f"  Progress: {idx + 1}/{total} files validated...")
        
        return df.iloc[valid_indices].reset_index(drop=True)
    
    def _validate_dataset_lenient(self, df: pd.DataFrame) -> pd.DataFrame:
        """Lenient validation - just check file existence and basic OpenCV loading"""
        print("🔄 Performing lenient validation (file existence + basic OpenCV)...")
        
        valid_indices = []
        total = len(df)
        
        for idx, row in df.iterrows():
            video_path = row['video_path']
            
            # Lenient criteria: file exists and OpenCV can open it
            if (os.path.exists(video_path) and 
                os.path.getsize(video_path) > 1000 and
                self._basic_opencv_check(video_path)):
                valid_indices.append(idx)
                self.stats['valid'] += 1
            else:
                self.stats['invalid'] += 1
            
            self.stats['total'] += 1
            
            if (idx + 1) % 500 == 0:
                print(f"  Progress: {idx + 1}/{total} files validated...")
        
        return df.iloc[valid_indices].reset_index(drop=True)
    
    def _basic_opencv_check(self, video_path: str) -> bool:
        """Most basic OpenCV check - just try to open"""
        try:
            with ultimate_stderr_suppression():
                cap = cv2.VideoCapture(video_path)
                is_opened = cap.isOpened()
                cap.release()
                return is_opened
        except:
            return False
    
    def print_stats(self):
        """Print comprehensive validation statistics"""
        print(f"\n📊 Final validation statistics:")
        print(f"  Total files: {self.stats['total']}")
        print(f"  ✅ Valid files: {self.stats['valid']}")
        print(f"  ❌ Invalid files: {self.stats['invalid']}")
        print(f"  📁 Not found: {self.stats['not_found']}")
        print(f"  📏 Too small: {self.stats['too_small']}")
        if self.ffprobe_available:
            print(f"  🔧 FFprobe success: {self.stats['ffprobe_success']}")
            print(f"  🔧 FFprobe failed: {self.stats['ffprobe_failed']}")
        print(f"  📹 OpenCV success: {self.stats['opencv_success']}")
        print(f"  📹 OpenCV failed: {self.stats['opencv_failed']}")
        if self.stats['total'] > 0:
            print(f"  📈 Overall success rate: {100 * self.stats['valid'] / self.stats['total']:.1f}%")

=== SMART MULTIMODAL CLASSIFIER - ADAPTIVE VALIDATION (FIXED) ===


In [6]:
# =============================================================================
# 2. ROBUST VIDEO LOADER (SAME AS BEFORE)
# =============================================================================

class RobustVideoLoader:
    """Robust video loader with better error handling"""
    
    def __init__(self, max_frames=8, image_size=224):
        self.max_frames = max_frames
        self.image_size = image_size
        self.video_mean = [0.485, 0.456, 0.406]
        self.video_std = [0.229, 0.224, 0.225]
        self.successful_loads = 0
        self.failed_loads = 0
        
        # Create diverse fallback videos
        self.fallback_videos = self._create_fallback_videos()
    
    def _create_fallback_videos(self):
        """Create multiple types of fallback videos"""
        fallbacks = {}
        
        # Black video
        black = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
        fallbacks['black'] = self._normalize_video(black)
        
        # Noise video
        noise = torch.randn(self.max_frames, 3, self.image_size, self.image_size) * 0.2
        fallbacks['noise'] = self._normalize_video(noise)
        
        # Simple pattern
        pattern = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
        for t in range(self.max_frames):
            for c in range(3):
                x = torch.arange(self.image_size).float() / self.image_size
                y = torch.arange(self.image_size).float() / self.image_size
                xx, yy = torch.meshgrid(x, y, indexing='ij')
                pattern[t, c] = (xx + yy + t / self.max_frames) % 1
        fallbacks['pattern'] = self._normalize_video(pattern)
        
        return fallbacks
    
    def _normalize_video(self, video_tensor):
        """Apply ImageNet normalization"""
        mean = torch.tensor(self.video_mean).view(1, 3, 1, 1)
        std = torch.tensor(self.video_std).view(1, 3, 1, 1)
        return (video_tensor - mean) / std
    
    def load_video_robust(self, video_path: str) -> torch.Tensor:
        """Load video with comprehensive error handling"""
        try:
            with ultimate_stderr_suppression():
                cap = None
                
                # Try multiple backends
                for backend in [cv2.CAP_FFMPEG, cv2.CAP_ANY]:
                    try:
                        cap = cv2.VideoCapture(video_path, backend)
                        if cap and cap.isOpened():
                            break
                    except:
                        continue
                
                if not cap or not cap.isOpened():
                    self.failed_loads += 1
                    return self._get_fallback_video()
                
                # Get video properties
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if total_frames <= 0:
                    total_frames = 30  # Assume 30 frames if unknown
                
                # Sample frames
                if total_frames <= self.max_frames:
                    frame_indices = list(range(total_frames)) + [total_frames-1] * (self.max_frames - total_frames)
                else:
                    frame_indices = np.linspace(0, total_frames-1, self.max_frames, dtype=int)
                
                frames = []
                successful_frames = 0
                
                for idx in frame_indices:
                    try:
                        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
                        ret, frame = cap.read()
                        
                        if ret and frame is not None and frame.size > 0:
                            # Process frame
                            frame = cv2.resize(frame, (self.image_size, self.image_size))
                            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                            frame = frame.astype(np.float32) / 255.0
                            
                            # Apply normalization
                            mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                            std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                            frame = (frame - mean) / std
                            
                            frames.append(frame)
                            successful_frames += 1
                        else:
                            # Use previous frame or create dummy
                            if frames:
                                frames.append(frames[-1])
                            else:
                                dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
                                mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                                std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                                frames.append((dummy - mean) / std)
                    except:
                        # Fallback frame
                        if frames:
                            frames.append(frames[-1])
                        else:
                            dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
                            mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                            std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                            frames.append((dummy - mean) / std)
                
                cap.release()
                
                # Ensure we have enough frames
                while len(frames) < self.max_frames:
                    if frames:
                        frames.append(frames[-1])
                    else:
                        dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
                        mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
                        std = np.array(self.video_std, dtype=np.float32).reshape(1, 1, 3)
                        frames.append((dummy - mean) / std)
                
                # Convert to tensor (T, H, W, C) -> (T, C, H, W)
                video_array = np.stack(frames[:self.max_frames])
                video_tensor = torch.from_numpy(video_array).float()
                video_tensor = video_tensor.permute(0, 3, 1, 2)  # (T, C, H, W)
                
                # Accept any video that loads (very lenient)
                self.successful_loads += 1
                return video_tensor
                
        except:
            self.failed_loads += 1
            return self._get_fallback_video()
    
    def _get_fallback_video(self):
        """Get diverse fallback videos"""
        fallback_types = ['black', 'noise', 'pattern']
        fallback_type = fallback_types[self.failed_loads % len(fallback_types)]
        return self.fallback_videos[fallback_type].clone()
    
    def get_stats(self):
        total = self.successful_loads + self.failed_loads
        if total > 0:
            print(f"📹 Video loading: {self.successful_loads}/{total} successful ({100*self.successful_loads/total:.1f}%)")


In [7]:
# =============================================================================
# 3-8. MODEL AND TRAINING CODE (SAME AS BEFORE)
# =============================================================================
import pickle

@dataclass
class SmartMultimodalConfig(PretrainedConfig):
    model_type = "smart_multimodal"
    
    def __init__(
        self,
        num_classes: int = 4,
        text_model_name: str = "bert-base-multilingual-cased",
        video_model_name: str = "facebook/timesformer-base-finetuned-k400",
        # hashtag_model_name: str = "BAAI/bge-m3",
        text_hidden_size: int = 768,
        video_hidden_size: int = 768,
        fusion_hidden_size: int = 256,
        classifier_hidden_size: int = 128,
        max_frames: int = 8,
        image_size: int = 224,
        dropout: float = 0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.text_model_name = text_model_name
        # self.hashtag_model_name = hashtag_model_name
        self.video_model_name = video_model_name
        self.text_hidden_size = text_hidden_size
        self.video_hidden_size = video_hidden_size
        self.fusion_hidden_size = fusion_hidden_size
        self.classifier_hidden_size = classifier_hidden_size
        self.max_frames = max_frames
        self.image_size = image_size
        self.dropout = dropout

import torch
from torch import nn
import torch.nn.functional as F

import math

from einops import rearrange

class LayerNorm(nn.Module):
    def __init__(self, size, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.eps = eps
        self.a_2 = nn.Parameter(torch.ones(size))
        self.b_2 = nn.Parameter(torch.zeros(size))

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

class FC(nn.Module):
    def __init__(self, in_size, out_size, dropout_r=0., use_relu=True):
        super(FC, self).__init__()
        self.dropout_r = dropout_r
        self.use_relu = use_relu
        self.linear = nn.Linear(in_size, out_size)
        if use_relu:
            self.relu = nn.GELU()
        if dropout_r > 0:
            self.dropout = nn.Dropout(dropout_r)

    def forward(self, x):
        x = self.linear(x)
        if self.use_relu:
            x = self.relu(x)
        if self.dropout_r > 0:
            x = self.dropout(x)
        return x

class MLP(nn.Module):
    def __init__(self, in_size, mid_size, out_size, dropout_r=0.1, use_relu=True):
        super().__init__()
        self.fc = FC(in_size, mid_size, dropout_r, use_relu)
        self.linear = nn.Linear(mid_size, out_size)

    def forward(self, x):
        return self.linear(self.fc(x))

class SelfAttention(nn.Module):
    def __init__(
        self,
        *,
        dim,
        heads = 8,
        dim_head = 64,
        dropout = 0.
    ):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5

        self.to_q = nn.Linear(dim, inner_dim, bias = False)
        self.to_k = nn.Linear(dim, inner_dim, bias = False)
        self.to_v = nn.Linear(dim, inner_dim, bias = False)
        self.to_out = nn.Linear(inner_dim, dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        b, n, _, h = *x.shape, self.heads
        q, k, v = self.to_q(x), self.to_k(x), self.to_v(x)

        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = h), (q, k, v))

        sim = torch.einsum('b h i d, b h j d -> b h i j', q, k)
        sim = sim * self.scale

        # mask

        mask_value = -torch.finfo(sim.dtype).max

        if mask is not None:  # mask: [B, N]
            mask = mask.unsqueeze(1).unsqueeze(2)  # [B,1,1,N]
            sim = sim.masked_fill(~mask, -torch.finfo(sim.dtype).max)

        # attention

        attn = sim.softmax(dim = -1)
        attn = self.dropout(attn)

        # aggregate

        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        
        # merge heads

        out = rearrange(out, 'b h n d -> b n (h d)')
        
        # combine heads and linear output
        
        return self.to_out(out)

class SA(nn.Module):

    def __init__(self, feat_dim, heads=8):
        super(SA, self).__init__()
        self.SelfAttention = SelfAttention(dim=feat_dim, heads=heads)
        self.norm1 = LayerNorm(feat_dim)
        self.norm2 = LayerNorm(feat_dim)
        self.ffn = MLP(in_size= feat_dim,
                       mid_size= feat_dim*2,
                       out_size= feat_dim,
                       dropout_r= 0.1,
                       use_relu=True)
    def forward(self, x, mask):
        x = self.norm1(x + 
            self.SelfAttention(x, mask)
        )
        x = self.norm2(x + 
            self.ffn(x)
        )
        return x

class SmartMultimodalModel(PreTrainedModel):
    config_class = SmartMultimodalConfig
    
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        
        print(f"🔤 Initializing text model: {config.text_model_name}")
        self.text_encoder = AutoModel.from_pretrained(config.text_model_name)
        
        print(f"🎬 Initializing video model: {config.video_model_name}")
        self.video_encoder = TimesformerForVideoClassification.from_pretrained(
            config.video_model_name,
            num_labels=config.num_classes,
            ignore_mismatched_sizes=True
        )
        # print(f"🔗 Initializing hashtag encoder: {config.hashtag_model_name}")
        # self.hashtag_encoder = SentenceTransformer(config.hashtag_model_name)
        

        # FREEZE BACKBONES_____________________________________________________________________
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        
        for param in self.video_encoder.parameters():
            param.requires_grad = False
        
        print("Text and Video encoders are frozen.")

        
        # Get actual dimensions
        video_dim = self.video_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size
        
        self.config.video_hidden_size = video_dim
        self.config.text_hidden_size = text_dim
        
        print(f"📐 Model dimensions - Text: {text_dim}, Video: {video_dim}")
        
        # Projection layers
        self.text_projection = nn.Sequential(
            nn.Linear(text_dim, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )
        
        self.video_projection = nn.Sequential(
            nn.Linear(video_dim, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

        self.hashtag_projection = nn.Sequential(
            # Input dim (B, 4)
            nn.Linear(4, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

        self.enc = nn.ModuleList([
            SA(feat_dim=config.fusion_hidden_size, heads=8)
            for _ in range(4)
        ])
        
        # Fusion layers
        self.fusion = nn.Sequential(
            nn.Linear(config.fusion_hidden_size, config.classifier_hidden_size),
            nn.LayerNorm(config.classifier_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )
        
        self.classifier = nn.Linear(config.classifier_hidden_size, config.num_classes)
        
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"🧠 Model: {total_params:,} total, {trainable_params:,} trainable")
    
    def forward(self, pixel_values, input_ids, attention_mask, hashtags=None, labels=None, **kwargs):
        # Text encoding
        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_features = text_outputs.last_hidden_state
        text_features = self.text_projection(text_features)
        text_mask = attention_mask.bool() 
        
        # Video encoding with silent error handling
        try:
            # Fix tensor dimensions: (B, T, C, H, W) -> (B, T, C, H, W) (TimeSformer format)
            batch_size, num_frames, channels, height, width = pixel_values.shape
            
            video_outputs = self.video_encoder.timesformer(pixel_values)
            video_features = video_outputs.last_hidden_state
            
        except:
            # Silent fallback
            batch_size = pixel_values.size(0)
            video_features = torch.zeros(
                batch_size, self.config.video_hidden_size,
                dtype=text_features.dtype, device=text_features.device
            )
        video_mask = torch.ones(
            video_features.size(0),  # batch size
            video_features.size(1),  # số token thực tế từ TimeSformer
            dtype=torch.bool,
            device=video_features.device
        )
        video_features = self.video_projection(video_features)
        
        # HASHTAG ENCODING 
        if hashtags is None:
            hashtags = torch.zeros(input_ids.size(0), 4, device=input_ids.device, dtype=torch.float32)
        else:
            hashtags = hashtags.to(input_ids.device).float()

        hashtag_features = self.hashtag_projection(hashtags).unsqueeze(1)
        hashtag_mask = torch.ones(
            hashtag_features.size(0),
            1,
            dtype=torch.bool,
            device=hashtag_features.device
        )


        fused_features = torch.cat(
            [text_features, video_features, hashtag_features],
            dim=1
        )
        fused_mask = torch.cat(
            [text_mask, video_mask, hashtag_mask],
            dim=1
        )  # [B, L_total]
        
        for enc in self.enc:
            fused_features = enc(fused_features, mask=fused_mask)

        pooled = fused_features.mean(dim=1)
        fused_features = self.fusion(pooled)
        logits = self.classifier(fused_features)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=0.1)
            loss = loss_fct(logits, labels)
        
        return {'loss': loss, 'logits': logits}

class SmartDataManager:
    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.df = None
        self.classes = None
        self.class_to_id = None
        self.id_to_class = None
        self.num_classes = None
        
    def load_data(self):
        print("📂 Loading dataset...")
        self.df = pd.read_csv(self.csv_path).dropna(subset=['class_name'])
        
        KAGGLE_VIDEO_ROOT = "/kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset"

        def fix_kaggle_path(old_path):
            if not isinstance(old_path, str):
                return old_path
            
            # Lấy phần sau "Dataset/"
            if "Dataset/" in old_path:
                relative_part = old_path.split("Dataset/")[-1]
                return os.path.join(KAGGLE_VIDEO_ROOT, relative_part)
            
            return old_path
        
        self.df["video_path"] = self.df["video_path"].apply(fix_kaggle_path)

        
        # Fix video paths
        # self.df['video_path'] = self.df['video_path'].str.replace(
        #     r'^/kaggle/working/extra_tikharm/',
        #     '/kaggle/input/extra-dataset/',
        #     regex=True
        # )
        
        self.df['combined_text'] = self.df['combined_text'].fillna("")
        
        print(f"Dataset loaded: {len(self.df)} samples")
        print(f"Classes distribution:\n{self.df['class_name'].value_counts()}")
        self.classes = sorted(self.df['class_name'].unique())
        self.class_to_id = {cls: i for i, cls in enumerate(self.classes)}
        self.id_to_class = {i: cls for cls, i in self.class_to_id.items()}
        self.num_classes = len(self.classes)
        
        print(f"📊 Classes ({self.num_classes}): {self.classes}")
        
        # Print some sample paths to debug
        print(f"\n🔍 Sample video paths:")
        for i in range(min(5, len(self.df))):
            path = self.df.iloc[i]['video_path']
            exists = os.path.exists(path)
            print(f"  {path} - Exists: {exists}")
        
    def get_smart_splits(self, validator: SmartVideoValidator):
        """Get splits with smart validation"""
        print("🧹 Creating smart validated splits...")
        
        # Get original splits
        train_df = self.df[self.df['split'] == 'train'].reset_index(drop=True)
        val_df = self.df[self.df['split'] == 'val'].reset_index(drop=True)
        test_df = self.df[self.df['split'] == 'test'].reset_index(drop=True)

        # train_df = self.df[(self.df['split'] == 'train') & (self.df['original_dir'] != 'extra')].reset_index(drop=True)
        # val_df = self.df[(self.df['split'] == 'val') & (self.df['original_dir'] != 'extra')].reset_index(drop=True)
        # test_df = self.df[self.df['split'] == 'test'].reset_index(drop=True)
        
        print(f"Original splits - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
        
        # Use sample validation first
        print("Validating training set...")
        clean_train_df = validator.validate_dataset_sample(train_df, sample_size=100)
        if len(clean_train_df) < 50:
            print("⚠️ Too few training samples, using lenient mode for all...")
            clean_train_df = validator._validate_dataset_lenient(train_df)
        
        print("Validating validation set...")
        clean_val_df = validator.validate_dataset_sample(val_df, sample_size=50)
        if len(clean_val_df) < 10:
            clean_val_df = validator._validate_dataset_lenient(val_df)
        
        print("Validating test set...")
        clean_test_df = validator.validate_dataset_sample(test_df, sample_size=50)
        if len(clean_test_df) < 10:
            clean_test_df = validator._validate_dataset_lenient(test_df)
        
        validator.print_stats()
        
        print(f"\n✨ Final splits:")
        print(f"  🏋️ Train: {len(clean_train_df)} (was {len(train_df)})")
        print(f"  ✅ Val: {len(clean_val_df)} (was {len(val_df)})")
        print(f"  🧪 Test: {len(clean_test_df)} (was {len(test_df)})")
        
        return clean_train_df, clean_val_df, clean_test_df

class SmartDataset(Dataset):
    """Smart dataset with robust video loading AND TRACKING"""
    
    def __init__(self, dataframe, hashtag_vector_map, text_tokenizer, max_frames=8, image_size=224, max_text_length=64, dataset_name="unknown"):
        self.df = dataframe
        self.hashtag_vector_map = hashtag_vector_map
        self.text_tokenizer = text_tokenizer
        self.max_frames = max_frames
        self.image_size = image_size
        self.max_text_length = max_text_length
        self.dataset_name = dataset_name  # Track dataset name
        
        # FIXED: Store actual dataframe length at creation
        self.actual_length = len(self.df)
        
        self.video_loader = RobustVideoLoader(max_frames, image_size)
        
        print(f"🛡️ {self.dataset_name} dataset: {self.actual_length} samples (verified)")
        
        # Store indices for verification
        self.sample_indices = list(self.df.index)
        
    def __len__(self):
        # FIXED: Return actual dataframe length, not processed count
        return self.actual_length
    
    def __getitem__(self, idx):
        try:
            # FIXED: Use actual dataframe index 
            row = self.df.iloc[idx]
            
            # Process text
            text = str(row['combined_text']) if pd.notna(row['combined_text']) else ""
            if not text.strip():
                text = "[EMPTY]"
            
            text_inputs = self.text_tokenizer(
                text,
                max_length=self.max_text_length,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )

            # Extract and Process hashtag
            hashtags = row.get("hashtags", "")

            if isinstance(hashtags, str):
                hashtag_list = [h.strip() for h in hashtags.split(",") if h.strip()]
            else:
                hashtag_list = []
            
            vectors = []
            if len(hashtag_list) == 0:
                vec = torch.zeros(4)
                vectors.append(vec)
            else:               
                for h in hashtag_list:
                    key = h.replace("#", "")
                    if key in self.hashtag_vector_map:
                        vec = torch.tensor(self.hashtag_vector_map[key], dtype=torch.float32)
                        vectors.append(vec)
                        
            # Not contain in map    
            if len(vectors)==0:
                final_hashtag_vec = torch.zeros(4)
            else:
                final_hashtag_vec = torch.stack(vectors).max(dim=0).values
            
            # Load video robustly
            video_path = row['video_path']
            pixel_values = self.video_loader.load_video_robust(video_path)
            
            # Get label
            label = data_manager.class_to_id[row['class_name']]
            
            return {
                'pixel_values': pixel_values,
                'input_ids': text_inputs['input_ids'].squeeze(0),
                'attention_mask': text_inputs['attention_mask'].squeeze(0),
                'hashtag_vec': final_hashtag_vec,
                'labels': torch.tensor(label, dtype=torch.long)
            }
            
        except Exception as e:
            print(f"⚠️ Error at index {idx} in {self.dataset_name}: {e}")
            
            # Safe fallback
            dummy_video = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
            dummy_input_ids = torch.zeros(self.max_text_length, dtype=torch.long)
            dummy_attention_mask = torch.zeros(self.max_text_length, dtype=torch.long)
            dummy_hashtag_vec = torch.zeros(4)
            
            return {
                'pixel_values': dummy_video,
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'hashtag_vec': dummy_hashtag_vec,
                'labels': torch.tensor(0, dtype=torch.long)
            }
    
    def get_stats(self):
        print(f"📊 {self.dataset_name} stats:")
        print(f"  DataFrame length: {len(self.df)}")
        print(f"  Actual length: {self.actual_length}")
        self.video_loader.get_stats()

def compute_comprehensive_metrics(eval_pred: EvalPrediction) -> Dict[str, float]:
    """Compute all 4 key metrics"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    recall = recall_score(labels, predictions, average='macro', zero_division=0)
    precision = precision_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,
        'precision': precision
    }

# def smart_collate_fn(batch):
#     """Smart collate function"""
#     try:
#         return {
#             'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
#             'input_ids': torch.stack([item['input_ids'] for item in batch]),
#             'attention_mask': torch.stack([item['attention_mask'] for item in batch]),
#             'hashtags': [item['hashtag_vec'] for item in batch],
#             'labels': torch.stack([item['labels'] for item in batch])
#         }
#     except Exception as e:
#         print(f"⚠️ Collate error: {e}")
#         batch_size = len(batch)
#         return {
#             'pixel_values': torch.zeros(batch_size, 8, 3, 224, 224),
#             'input_ids': torch.zeros(batch_size, 64, dtype=torch.long),
#             'attention_mask': torch.zeros(batch_size, 64, dtype=torch.long),
            
#             'labels': torch.zeros(batch_size, dtype=torch.long)
#         }
        
def smart_collate_fn(batch):
    """Smart collate function"""
    try:
        return {
            'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
            'input_ids': torch.stack([item['input_ids'] for item in batch]),
            'attention_mask': torch.stack([item['attention_mask'] for item in batch]),    
            # stack tensor hashtag
            'hashtag_vec': torch.stack([item['hashtag_vec'] for item in batch]),   
            'labels': torch.stack([item['labels'] for item in batch])
        }

    except Exception as e:
        print(f"⚠️ Collate error: {e}")
        batch_size = len(batch)

        return {
            'pixel_values': torch.zeros(batch_size, 8, 3, 224, 224),
            'input_ids': torch.zeros(batch_size, 64, dtype=torch.long),
            'attention_mask': torch.zeros(batch_size, 64, dtype=torch.long),
            # thêm hashtags fallback đúng shape
            'hashtag_vec': torch.zeros(batch_size, 4, dtype=torch.float32),
            'labels': torch.zeros(batch_size, dtype=torch.long)
        }

In [8]:
import pickle
from dataclasses import dataclass
import numpy as np
@dataclass
class HashtagResult:
    raw_hashtag: str
    normalized_hashtag: str
    embedding: np.ndarray
    cluster: str
    meta_cluster: str
    cluster_vector: np.ndarray

In [ ]:
def main():
    """Main training function with FIXED data counting"""
    print("📦 Loading hashtag meta vectors from PKL...")

    with open("/kaggle/input/datasets/dnnamm/hashtag-classification-result/hashtag_classification_results.pkl", "rb") as f:
        pkl = pickle.load(f)

    # build lookup dict
    hashtag_vector_map = {}

    for item in pkl:
        key = item.raw_hashtag.replace("#", "")
        hashtag_vector_map[key] = torch.tensor(
            item.cluster_vector, dtype=torch.float32
        )

    print(f"✅ Loaded {len(hashtag_vector_map)} hashtag vectors")
    
    try:
        print("🚀 Starting smart adaptive training (FIXED VERSION)...")
        
        # Initialize data manager
        global data_manager
        data_manager = SmartDataManager("/kaggle/input/datasets/dnnamm/dataset-with-all-hashtag/dataset_with_all_hashtag.csv")
        data_manager.load_data()
        
        # Initialize smart validator (not strict)
        print("🔍 Initializing smart video validator...")
        validator = SmartVideoValidator(strict_mode=False)
        
        # Get smart validated data
        train_df, val_df, test_df = data_manager.get_smart_splits(validator)
        
     
        # VERIFY DATA SPLITS AGAIN
        print(f"\n🔍 VERIFICATION - Actual dataframe sizes:")
        print(f"  Train DF: {len(train_df)} rows")
        print(f"  Val DF: {len(val_df)} rows") 
        print(f"  Test DF: {len(test_df)} rows")
        
        # Check data sufficiency
        if len(train_df) < 50:
            print("⚠️ Very few training samples!")
            print("🔄 Proceeding with available data...")
        
        # Initialize models
        print("🧠 Initializing models...")
        text_model_name = "bert-base-multilingual-cased"
        text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        
        config = SmartMultimodalConfig(
            num_classes=data_manager.num_classes,
            text_model_name=text_model_name,
            max_frames=8,
            image_size=224,
            dropout=0.1
        )
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"💻 Using device: {device}")
        
        model = SmartMultimodalModel(config)
        model.to(device)
        
        # Create smart datasets WITH PROPER TRACKING
        print("📊 Creating smart datasets with tracking...")
        train_dataset = SmartDataset(train_df, hashtag_vector_map, text_tokenizer, dataset_name="TRAIN")
        val_dataset = SmartDataset(val_df, hashtag_vector_map, text_tokenizer, dataset_name="VAL")
        test_dataset = SmartDataset(test_df, hashtag_vector_map, text_tokenizer, dataset_name="TEST")
        
        # VERIFY DATASET LENGTHS
        print(f"\n🎯 FINAL DATASET VERIFICATION:")
        print(f"  Train dataset: {len(train_dataset)} samples")
        print(f"  Val dataset: {len(val_dataset)} samples")
        print(f"  Test dataset: {len(test_dataset)} samples")
        
        # Test sample loading
        print("🧪 Testing sample loading...")
        sample = train_dataset[0]
        print(f"  ✅ Video shape: {sample['pixel_values'].shape}")
        print(f"  ✅ Text shape: {sample['input_ids'].shape}")
        print(f"  ✅ Hashtags shape: {sample['hashtag_vec']}")
        
        # Test forward pass
        model.eval()
        with torch.no_grad():
            batch = {k: v.unsqueeze(0).to(device) if isinstance(v, torch.Tensor) else v for k, v in sample.items()}
            batch["hashtag_vec"] = [batch["hashtag_vec"]]
            output = model(**batch)
            print(f"  ✅ Output shape: {output['logits'].shape}")

        OUTPUT_DIR = "/kaggle/working/checkpoints"
        FINAL_MODEL_DIR = "/kaggle/working/final_model"
        
        # Training setup with FIXED batch settings
        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs = 6,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            gradient_accumulation_steps=2,
            learning_rate=1e-4,
            weight_decay=0.01,
            warmup_steps=100,
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=100,
            save_strategy="steps",
            save_steps=100,
            load_best_model_at_end=True,
            metric_for_best_model="f1_score",
            fp16=True,
            dataloader_num_workers=0,
            remove_unused_columns=False,
            report_to="none",
            save_total_limit = 3,
            # FIXED: Don't drop last to preserve data count
            dataloader_drop_last=False,  # Changed from True to False
            max_grad_norm=1.0,
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=smart_collate_fn,
            compute_metrics=compute_comprehensive_metrics,
        )
        
        # Start training
        print("\n" + "="*80)
        print("🚀 STARTING SMART ADAPTIVE TRAINING (FIXED)!")
        print("="*80)
        
        start_time = time.time()

        
        # Train with error suppression
        last_checkpoint = None

        # AUTO RESUME LOGIC ______________________________________________________________________
        # if os.path.isdir(OUTPUT_DIR):
        #     checkpoints = [
        #         os.path.join(OUTPUT_DIR, d)
        #         for d in os.listdir(OUTPUT_DIR)
        #         if d.startswith("checkpoint-")
        #     ]
        #     if len(checkpoints) > 0:
        #         last_checkpoint = max(checkpoints, key=os.path.getctime)
        
        # if last_checkpoint:
        #     print(f"Resuming from checkpoint: {last_checkpoint}")
        #     trainer.train(resume_from_checkpoint=last_checkpoint)
        # else:
        #     print("Starting fresh training...")
        #     trainer.train()
        from transformers.trainer_utils import get_last_checkpoint
        if os.path.isdir(OUTPUT_DIR):
            last_checkpoint = get_last_checkpoint(OUTPUT_DIR)
        
        if last_checkpoint is not None:
            print(f"🔁 Resuming from checkpoint: {last_checkpoint}")
            trainer.train(resume_from_checkpoint=True)
        else:
            print("Starting fresh training...")
            trainer.train()

        
        training_time = time.time() - start_time
        
        print(f"\n🎉 TRAINING COMPLETED in {training_time:.1f}s!")
        
        # Save model
        trainer.save_model(FINAL_MODEL_DIR)
        print("💾 Model saved!")
        
        


        # Evaluation
        print("📊 Running evaluation...")
        eval_results = trainer.evaluate()
        
        print(f"\n🎯 VALIDATION RESULTS:")
        print(f"  Accuracy: {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")
        print(f"  F1-Score: {eval_results['eval_f1_score']:.4f}")
        print(f"  Recall: {eval_results['eval_recall']:.4f}")
        print(f"  Precision: {eval_results['eval_precision']:.4f}")
        
        # FIXED Test evaluation with manual prediction
        print("🧪 Running FIXED test evaluation...")
        
        # Use manual DataLoader to control exact sample count
        test_dataloader = DataLoader(
            test_dataset, 
            batch_size=8, 
            shuffle=False, 
            drop_last=False,  # Don't drop any samples
            collate_fn=smart_collate_fn
        )
        
        model.eval()
        all_predictions = []
        all_labels = []
        
        print(f"🔍 Processing test data in {len(test_dataloader)} batches...")
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(test_dataloader):
                # Move to device
                batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                
                # Get predictions
                outputs = model(**batch)
                predictions = torch.argmax(outputs['logits'], dim=1)
                
                all_predictions.extend(predictions.cpu().numpy())
                all_labels.extend(batch['labels'].cpu().numpy())
                
                if (batch_idx + 1) % 10 == 0:
                    print(f"  Processed batch {batch_idx + 1}/{len(test_dataloader)}")
        
        # Convert to numpy arrays
        y_pred = np.array(all_predictions)
        y_true = np.array(all_labels)
        
        print(f"\n🎯 FINAL TEST DATA VERIFICATION:")
        print(f"  Expected test samples: {len(test_dataset)}")
        print(f"  Actual predictions: {len(y_pred)}")
        print(f"  Actual labels: {len(y_true)}")
        print(f"  Data consistency: {'✅ CONSISTENT' if len(y_pred) == len(test_dataset) else '❌ INCONSISTENT'}")
        
        # Test metrics
        test_accuracy = accuracy_score(y_true, y_pred)
        test_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        test_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        
        print(f"\n🏆 FINAL TEST RESULTS (VERIFIED):")
        print(f"  Test samples: {len(y_true)}")
        print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
        print(f"  F1-Score: {test_f1:.4f}")
        print(f"  Recall: {test_recall:.4f}")
        print(f"  Precision: {test_precision:.4f}")
        
        # Classification report
        print(f"\n📋 CLASSIFICATION REPORT (VERIFIED):")
        print("-" * 60)
        class_report = classification_report(
            y_true, y_pred, 
            target_names=data_manager.classes,
            digits=4
        )
        print(class_report)
        
        # Save results
        import json
        results = {
            'model_type': 'Smart_Adaptive_Multimodal_FIXED',
            'data_verification': {
                'train_samples': len(train_dataset),
                'val_samples': len(val_dataset),
                'test_samples': len(test_dataset),
                'test_predictions': len(y_pred),
                'data_consistent': len(y_pred) == len(test_dataset)
            },
            'validation_metrics': {
                'accuracy': eval_results['eval_accuracy'],
                'f1_score': eval_results['eval_f1_score'],
                'recall': eval_results['eval_recall'],
                'precision': eval_results['eval_precision']
            },
            'test_metrics': {
                'accuracy': test_accuracy,
                'f1_score': test_f1,
                'recall': test_recall,
                'precision': test_precision
            },
            'training_time_seconds': training_time,
            'validation_stats': validator.stats,
            'dataset_sizes': {
                'train': len(train_dataset),
                'val': len(val_dataset),
                'test': len(test_dataset)
            }
        }
        
        with open('smart_results_fixed.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print("💾 Results saved!")
        
        # Final statistics
        print("\n📈 Final statistics:")
        train_dataset.get_stats()
        val_dataset.get_stats() 
        test_dataset.get_stats()
        
        print("\n" + "="*80)
        print("🎊 SMART TRAINING COMPLETED SUCCESSFULLY (FIXED)!")
        print("✅ DATA COUNTING ISSUES RESOLVED!")
        print("📊 ALL METRICS CALCULATED CORRECTLY!")
        print("="*80)

        
        # SAVE FULL HYPERPARAMETERS______________________________________________________________________
        full_config = {
            "training_args": trainer.args.to_dict(),
            "model_config": model.config.to_dict() if hasattr(model, "config") else {},
        }
        
        with open("/kaggle/working/experiment_config.json", "w") as f:
            json.dump(full_config, f, indent=4)


        # SAVE TRAINING HISTORY______________________________________________________________________
        with open("/kaggle/working/training_history.json", "w") as f:
            json.dump(trainer.state.log_history, f, indent=4)


        # SAVE FINAL METRICS______________________________________________________________________
        metrics = trainer.evaluate()
        
        with open("/kaggle/working/final_metrics.json", "w") as f:
            json.dump(metrics, f, indent=4)
        
        # SAVE ENVIRONMENT INFO______________________________________________________________________ 
        env_info = {
            "torch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        }
        
        with open("/kaggle/working/environment_info.json", "w") as f:
            json.dump(env_info, f, indent=4)
        
        print("✅ Everything saved safely in /kaggle/working/")
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        cleanup_memory()

In [ ]:
main()

In [ ]:
# import pandas as pd

# test = pd.read_csv('/kaggle/input/datasets/dnnamm/dataset-with-all-hashtag/dataset_with_all_hashtag.csv')

# old_prefix = "/kaggle/working/extra_tikharm"
# new_prefix = "/kaggle/input/datasets/kusnguyen/extra-dataset"

# # Chỉ thay với dòng có original_dir = 'extra'
# mask = test['original_dir'] == 'extra'

# test.loc[mask, 'video_path'] = test.loc[mask, 'video_path'].str.replace(
#     old_prefix, new_prefix, regex=False
# )

# # Check lại
# print(test[mask]['video_path'].head())

In [ ]:
# test.to_csv('/kaggle/working/dataset_with_all_hashtag.csv', index=False)

In [ ]:
# import shutil
# import os
# SOURCE_CKPT = "/kaggle/input/datasets/ziaxxx/cp-250/checkpoint-250"
# DEST_DIR = "/kaggle/working/checkpoints/checkpoint-2250"

# if not os.path.exists(DEST_DIR):
#     print("📦 Copying checkpoint to working directory...")
#     shutil.copytree(SOURCE_CKPT, DEST_DIR)

In [ ]:
# import torch
# capability = torch.cuda.get_device_capability(0)
# arch_list = torch.cuda.get_arch_list()

# print(f"Kiến trúc P100: {capability}")
# print(f"Hỗ trợ hiện tại: {arch_list}")

# if 'sm_60' in arch_list:
#     print("✅ Tuyệt vời! sm_60 đã có trong danh sách. Bạn có thể train ngay.")
# else:
#     print("❌ Vẫn chưa thấy sm_60. Có thể môi trường pre-installed đang bị xung đột.")

In [ ]:
# # import pandas as pd
# test = pd.read_csv('/kaggle/input/datasets/dnnamm/dataset-with-all-hashtag/dataset_with_all_hashtag.csv')
# print(test[test['original_dir'] == 'extra']['video_path'])

In [ ]:
import os

checkpoint_dir = "/kaggle/working/checkpoints"
if os.path.exists(checkpoint_dir):
    files = os.listdir(checkpoint_dir)
    print(f"📁 Thư mục checkpoints chứa: {files}")
    if len(files) == 0:
        print("Empty! Đúng như dự đoán, chưa có checkpoint nào được lưu vì save_steps=100.")
else:
    print("❌ Thư mục checkpoints thậm chí còn chưa được tạo ra!")

## Error analyst

Analyse with hashtags

In [29]:
# ============================================================
# ERROR ANALYSIS MODULE
# ============================================================
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score
)
from torch.utils.data import DataLoader
import torch


# ─────────────────────────────────────────────
# 1. Load model từ final_model dir
# ─────────────────────────────────────────────
def load_model_for_analysis(
    final_model_dir: str,
    config: SmartMultimodalConfig,
    device: torch.device,
) -> SmartMultimodalModel:
    """Load best model từ final_model dir (do trainer.save_model() lưu)."""
    print(f"📂 Loading model from: {final_model_dir}")

    if not os.path.isdir(final_model_dir):
        raise FileNotFoundError(f"Final model dir not found: {final_model_dir}")

    model = SmartMultimodalModel(config)

    safe_path = os.path.join(final_model_dir, "model.safetensors")
    bin_path  = os.path.join(final_model_dir, "pytorch_model.bin")

    if os.path.exists(safe_path):
        from safetensors.torch import load_file
        state_dict = load_file(safe_path, device=str(device))
        print("  ✅ Loaded from model.safetensors")
    elif os.path.exists(bin_path):
        state_dict = torch.load(bin_path, map_location=device)
        print("  ✅ Loaded from pytorch_model.bin")
    else:
        raise FileNotFoundError(
            f"No weight file found in {final_model_dir}\n"
            f"  Expected: {safe_path}\n       or: {bin_path}"
        )

    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    print("✅ Model ready for inference")
    return model


# ─────────────────────────────────────────────
# 2. Collect predictions
# ─────────────────────────────────────────────
def collect_predictions(
    model: SmartMultimodalModel,
    dataset,
    dataframe: pd.DataFrame,
    device: torch.device,
    batch_size: int = 8,
    collate_fn=None,
) -> pd.DataFrame:
    """
    Chạy inference toàn bộ dataset, trả về DataFrame có thêm cột:
    true_idx, pred_idx, confidence, is_wrong, prob_vector
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }
            outputs = model(**batch)
            logits  = outputs["logits"]
            probs   = torch.softmax(logits, dim=-1).cpu().numpy()
            preds   = logits.argmax(dim=-1).cpu().numpy()
            labels  = batch["labels"].cpu().numpy()

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels)

            if (batch_idx + 1) % 20 == 0:
                print(f"  Processed {batch_idx + 1}/{len(loader)} batches...")

    result_df = dataframe.copy().reset_index(drop=True)
    result_df["true_idx"]   = all_labels
    result_df["pred_idx"]   = all_preds
    result_df["confidence"] = [p[pred] for p, pred in zip(all_probs, all_preds)]
    result_df["is_wrong"]   = result_df["true_idx"] != result_df["pred_idx"]
    result_df["prob_vector"] = [json.dumps(p.tolist()) for p in all_probs]

    print(f"\n📊 Prediction summary:")
    print(f"  Total samples : {len(result_df)}")
    print(f"  Correct       : {(~result_df['is_wrong']).sum()}")
    print(f"  Wrong         : {result_df['is_wrong'].sum()}")
    print(f"  Accuracy      : {(~result_df['is_wrong']).mean()*100:.2f}%")
    return result_df


# ─────────────────────────────────────────────
# 3. Confusion matrix
# ─────────────────────────────────────────────
def plot_confusion_matrix(
    result_df: pd.DataFrame,
    class_names: list,
    save_dir: str,
) -> np.ndarray:
    y_true = result_df["true_idx"].values
    y_pred = result_df["pred_idx"].values

    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(max(12, len(class_names) * 1.2),
                 max(8,  len(class_names) * 0.9))
    )
    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Confusion matrix (count)", "Confusion matrix (normalized)"],
        ["d", ".2f"],
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.4, ax=ax,
        )
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Predicted", fontsize=10)
        ax.set_ylabel("True", fontsize=10)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.tick_params(axis="y", rotation=0,  labelsize=8)

    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"💾 Confusion matrix saved → {path}")

    # Per-class stats
    per_class = pd.DataFrame({
        "class":     class_names,
        "n_samples": cm.sum(axis=1),
        "n_correct": cm.diagonal(),
        "recall":    cm_norm.diagonal(),
    })
    per_class["error_rate"] = 1 - per_class["recall"]
    per_class = per_class.sort_values("error_rate", ascending=False)

    print("\n🔍 Per-class error rates (worst first):")
    print(per_class.to_string(index=False))
    per_class.to_csv(os.path.join(save_dir, "per_class_stats.csv"), index=False)

    return cm


# ─────────────────────────────────────────────
# 4. Lưu wrong samples
# ─────────────────────────────────────────────
def save_wrong_samples(
    result_df: pd.DataFrame,
    class_names: list,
    save_dir: str,
    top_k: int = 200,
) -> pd.DataFrame:
    idx2cls  = {i: c for i, c in enumerate(class_names)}
    wrong_df = result_df[result_df["is_wrong"]].copy()

    wrong_df["true_label"] = wrong_df["true_idx"].map(idx2cls)
    wrong_df["pred_label"] = wrong_df["pred_idx"].map(idx2cls)
    wrong_df["error_pair"] = wrong_df["true_label"] + " → " + wrong_df["pred_label"]

    # Model tự tin cao nhưng vẫn sai = nguy hiểm nhất → lên đầu
    wrong_df = wrong_df.sort_values("confidence", ascending=False)

    csv_path  = os.path.join(save_dir, "wrong_predictions.csv")
    json_path = os.path.join(save_dir, "wrong_predictions.json")

    wrong_df.drop(columns=["prob_vector"], errors="ignore").to_csv(csv_path, index=False)
    wrong_df.head(top_k).drop(columns=["prob_vector"], errors="ignore").to_json(
        json_path, orient="records", force_ascii=False, indent=2
    )
    print(f"💾 Wrong samples saved → {csv_path} ({len(wrong_df)} rows)")

    # Top confused pairs
    pair_counts = wrong_df["error_pair"].value_counts().head(20)
    print("\n🔴 Top confused class pairs:")
    print(pair_counts.to_string())
    pair_counts.to_csv(os.path.join(save_dir, "top_confused_pairs.csv"), header=["count"])

    # Bar chart top confused pairs
    fig, ax = plt.subplots(figsize=(10, max(4, len(pair_counts) * 0.4)))
    pair_counts[::-1].plot(kind="barh", ax=ax, color="salmon")
    ax.set_title("Top confused class pairs", fontsize=12)
    ax.set_xlabel("Count", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "top_confused_pairs.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return wrong_df


# # ─────────────────────────────────────────────
# # 5. Hashtag error analysis (không cluster)
# # ─────────────────────────────────────────────
# def hashtag_error_analysis(
#     result_df: pd.DataFrame,
#     class_names: list,
#     save_dir: str,
#     hashtag_col: str = "hashtags",
#     top_n: int = 30,
# ):
#     """
#     Tính error rate theo từng hashtag và vẽ:
#       - Bar chart top hashtag có error rate cao nhất
#       - Heatmap hashtag × predicted_class (top hashtag bị sai)
#     """
#     if hashtag_col not in result_df.columns:
#         print(f"⚠️  Column '{hashtag_col}' not found — skipping hashtag analysis")
#         return

#     # Explode: mỗi hàng 1 hashtag
#     exploded = result_df.explode(hashtag_col).copy()
#     exploded[hashtag_col] = (
#         exploded[hashtag_col]
#         .astype(str)
#         .str.replace("#", "", regex=False)
#         .str.lower()
#         .str.strip()
#     )
#     exploded = exploded[
#         exploded[hashtag_col].notna()
#         & (exploded[hashtag_col] != "")
#         & (exploded[hashtag_col] != "nan")
#     ]

#     # ── 5a. Per-hashtag error rate ──
#     ht_stats = (
#         exploded.groupby(hashtag_col)
#         .agg(
#             n_samples=("is_wrong", "count"),
#             n_errors =("is_wrong", "sum"),
#             avg_conf =("confidence", "mean"),
#         )
#         .assign(error_rate=lambda d: d["n_errors"] / d["n_samples"])
#         .query("n_samples >= 3")
#         .sort_values("error_rate", ascending=False)
#         .reset_index()
#     )

#     ht_stats.to_csv(os.path.join(save_dir, "hashtag_error_rates.csv"), index=False)
#     print(f"\n🏷️  Top {top_n} hashtags by error rate (min 3 samples):")
#     print(ht_stats.head(top_n).to_string(index=False))

#     # ── 5b. Bar chart top hashtags ──
#     top_ht = ht_stats.head(top_n)
#     fig, ax = plt.subplots(figsize=(10, max(5, len(top_ht) * 0.38)))
#     colors  = plt.cm.RdYlGn_r(top_ht["error_rate"].values)
#     ax.barh(top_ht[hashtag_col][::-1], top_ht["error_rate"][::-1], color=colors[::-1])
#     ax.set_xlabel("Error rate", fontsize=10)
#     ax.set_title(f"Top {top_n} hashtags by error rate", fontsize=12)
#     ax.set_xlim(0, 1)
#     for spine in ["top", "right"]:
#         ax.spines[spine].set_visible(False)
#     # Annotate n_samples
#     for i, (_, row) in enumerate(top_ht[::-1].iterrows()):
#         ax.text(
#             row["error_rate"] + 0.01, i,
#             f"n={int(row['n_samples'])}",
#             va="center", fontsize=7, color="gray",
#         )
#     plt.tight_layout()
#     plt.savefig(os.path.join(save_dir, "hashtag_error_rates.png"), dpi=150, bbox_inches="tight")
#     plt.close()
#     print(f"💾 Hashtag error bar chart saved")

#     # ── 5c. Heatmap: top hashtag × predicted class (chỉ wrong samples) ──
#     idx2cls  = {i: c for i, c in enumerate(class_names)}
#     wrong_exp = exploded[exploded["is_wrong"]].copy()
#     wrong_exp["pred_label"] = wrong_exp["pred_idx"].map(idx2cls)
#     wrong_exp["true_label"] = wrong_exp["true_idx"].map(idx2cls)

#     # Lấy top hashtag bị sai nhiều nhất (theo số lần xuất hiện trong wrong set)
#     top_wrong_tags = (
#         wrong_exp[hashtag_col]
#         .value_counts()
#         .head(top_n)
#         .index.tolist()
#     )
#     wrong_top = wrong_exp[wrong_exp[hashtag_col].isin(top_wrong_tags)]

#     if len(wrong_top) > 0:
#         pivot = (
#             wrong_top.groupby([hashtag_col, "pred_label"])
#             .size()
#             .unstack(fill_value=0)
#         )
#         # Sắp xếp hàng theo tổng lỗi giảm dần
#         pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

#         fig, ax = plt.subplots(
#             figsize=(max(10, len(class_names) * 1.1),
#                      max(6, len(pivot) * 0.45))
#         )
#         sns.heatmap(
#             pivot, annot=True, fmt="d", cmap="YlOrRd",
#             linewidths=0.3, ax=ax,
#         )
#         ax.set_title(
#             f"Wrong predictions: top hashtags × predicted class\n"
#             f"(top {top_n} most-errored hashtags)",
#             fontsize=11,
#         )
#         ax.set_xlabel("Predicted class", fontsize=10)
#         ax.set_ylabel("Hashtag", fontsize=10)
#         ax.tick_params(axis="x", rotation=45, labelsize=8)
#         ax.tick_params(axis="y", rotation=0,  labelsize=8)
#         plt.tight_layout()
#         hm_path = os.path.join(save_dir, "hashtag_vs_class_heatmap.png")
#         plt.savefig(hm_path, dpi=150, bbox_inches="tight")
#         plt.close()
#         print(f"💾 Hashtag × class heatmap saved → {hm_path}")

#     # ── 5d. Hashtag confusion: true vs pred breakdown per hashtag ──
#     if len(wrong_top) > 0:
#         error_detail = (
#             wrong_top.groupby([hashtag_col, "true_label", "pred_label"])
#             .size()
#             .reset_index(name="count")
#             .sort_values(["hashtag_col" if hashtag_col == "hashtag_col" else hashtag_col, "count"],
#                          ascending=[True, False])
#         )
#         error_detail.to_csv(
#             os.path.join(save_dir, "hashtag_error_detail.csv"), index=False
#         )
#         print(f"💾 Hashtag error detail saved")


# ─────────────────────────────────────────────
# 6. Master runner
# ─────────────────────────────────────────────
def run_error_analysis(
    final_model_dir: str = "/kaggle/working/final_model",
    analysis_dir:    str = "/kaggle/working/error_analysis",
    hashtag_pkl:     str = "path_to_your.pkl",
    hashtag_col:     str = "hashtags",
    batch_size:      int = 8,
    top_n:           int = 30,
):
    os.makedirs(analysis_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Load hashtag vectors ──
    print("📦 Loading hashtag vectors...")
    with open(hashtag_pkl, "rb") as f:
        pkl = pickle.load(f)
    hashtag_vector_map = {
        item.raw_hashtag.replace("#", ""): torch.tensor(
            item.cluster_vector, dtype=torch.float32
        )
        for item in pkl
    }
    print(f"  {len(hashtag_vector_map)} hashtag vectors loaded")

    # ── Load data ──
    print("\n📂 Loading dataset...")
    global data_manager
    data_manager = SmartDataManager("/kaggle/input/datasets/dnnamm/dataset-with-all-hashtag/dataset_with_all_hashtag.csv")
    data_manager.load_data()

    tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
    config    = SmartMultimodalConfig(
        num_classes     = data_manager.num_classes,
        text_model_name = "bert-base-multilingual-cased",
        max_frames      = 8,
        image_size      = 224,
        dropout         = 0.1,
    )

    # ── Load best model từ final_model ──
    model = load_model_for_analysis(final_model_dir, config, device)

    # ── Build test split ──
    validator    = SmartVideoValidator(strict_mode=False)
    _, _, test_df = data_manager.get_smart_splits(validator)
    test_dataset = SmartDataset(
        test_df, hashtag_vector_map, tokenizer, dataset_name="TEST"
    )
    print(f"\n📊 Test set: {len(test_dataset)} samples | {data_manager.num_classes} classes")

    # ── Collect predictions ──
    result_df = collect_predictions(
        model, test_dataset, test_df, device,
        batch_size=batch_size, collate_fn=smart_collate_fn,
    )

    # ── Confusion matrix ──
    print("\n🔢 Computing confusion matrix...")
    cm = plot_confusion_matrix(result_df, data_manager.classes, analysis_dir)

    # ── Wrong samples ──
    print("\n💾 Saving wrong predictions...")
    wrong_df = save_wrong_samples(result_df, data_manager.classes, analysis_dir)

    # # ── Hashtag error analysis ──
    # print("\n🏷️  Running hashtag error analysis...")
    # hashtag_error_analysis(
    #     result_df, data_manager.classes, analysis_dir,
    #     hashtag_col=hashtag_col, top_n=top_n,
    # )

    # ── Classification report ──
    y_true = result_df["true_idx"].values
    y_pred = result_df["pred_idx"].values
    print(f"\n📋 Classification report:")
    print("-" * 60)
    print(classification_report(y_true, y_pred, target_names=data_manager.classes, digits=4, zero_division=0))

    # ── Final summary JSON ──
    summary = {
        "final_model_dir": final_model_dir,
        "total_samples":   int(len(result_df)),
        "total_errors":    int(result_df["is_wrong"].sum()),
        "accuracy":        float(accuracy_score(y_true, y_pred)),
        "f1_macro":        float(f1_score(y_true, y_pred, average="macro",  zero_division=0)),
        "f1_weighted":     float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "classification_report": classification_report(
            y_true, y_pred,
            target_names=data_manager.classes,
            output_dict=True,
            zero_division=0,
        ),
    }
    summary_path = os.path.join(analysis_dir, "error_analysis_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 60)
    print("✅ ERROR ANALYSIS COMPLETE")
    print(f"   Model       : {final_model_dir}")
    print(f"   Accuracy    : {summary['accuracy']*100:.2f}%")
    print(f"   F1 macro    : {summary['f1_macro']:.4f}")
    print(f"   F1 weighted : {summary['f1_weighted']:.4f}")
    print(f"   Errors      : {summary['total_errors']} / {summary['total_samples']}")
    print(f"   Outputs  →  {analysis_dir}/")
    print("=" * 60)

    return result_df, wrong_df, summary

In [30]:
# ─────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────
if __name__ == "__main__":
    result_df, wrong_df, summary = run_error_analysis(
        final_model_dir = "/kaggle/input/models/dnnamm/mguard-with-hashtag-classifiers/pytorch/default/1/final_model",
        analysis_dir    = "/kaggle/working/error_analysis",
        hashtag_pkl     = "/kaggle/input/datasets/dnnamm/hashtag-classification-result/hashtag_classification_results.pkl",
        hashtag_col     = "hashtags",   # tên cột chứa list hashtag trong dataframe
        batch_size      = 8,
        top_n           = 30,           # số hashtag hiển thị trong chart/heatmap
    )

📦 Loading hashtag vectors...
  10315 hashtag vectors loaded

📂 Loading dataset...
📂 Loading dataset...
Dataset loaded: 4723 samples
Classes distribution:
class_name
Safe               1248
Harmful Content    1193
Adult Content      1141
Suicide            1141
Name: count, dtype: int64
📊 Classes (4): ['Adult Content', 'Harmful Content', 'Safe', 'Suicide']

🔍 Sample video paths:
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/jaypark_foryou_7339639860354911493.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/1life.anthonyblane_7368846422709521671.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/loungebarvip_7252268249243454725.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/phangphong111_7287936860796620039.mp4 - Exists: True
  /kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset/train/Adult Content/girl_xi

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🎬 Initializing video model: facebook/timesformer-base-finetuned-k400


Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

TimesformerForVideoClassification LOAD REPORT from: facebook/timesformer-base-finetuned-k400
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Text and Video encoders are frozen.
📐 Model dimensions - Text: 768, Video: 768
🧠 Model: 302,697,864 total, 3,582,596 trainable
  ✅ Loaded from model.safetensors
✅ Model ready for inference
⚠️ FFprobe not available - will use OpenCV-only validation
📋 Validator initialized:
  FFprobe available: False
  Strict mode: False
🧹 Creating smart validated splits...
Original splits - Train: 3418, Val: 515, Test: 790
Validating training set...
🧪 Testing validation on 100 samples...
📊 Sample validation results:
  Tested: 100 files
  Valid: 100
  Success rate: 100.0%
🔍 Performing normal video validation...
  Progress: 500/3418 files validated...
  Progress: 1000/3418 files validated...
  Progress: 1500/3418 files validated...
  Progress: 2000/3418 files validated...
  Progress: 2500/3418 files validated...
  Progress: 3000/3418 files validated...
Validating validation set...
🧪 Testing validation on 50 samples...
📊 Sample validation results:
  Tested: 50 files
  Valid: 50
  Success rate: 100.0%
🔍 Per

In [32]:
import shutil

shutil.make_archive(
    "/kaggle/working/error_analysis",  # tên file zip
    'zip',
    "/kaggle/working/error_analysis"   # folder cần zip
)

'/kaggle/working/error_analysis.zip'